# Personal Knowledge Assistant
**Agentic AI Course · Final Project · Bano Qabil, Lahore**

Researcher (apne documents mein search) → Verifier (live web se check) → Writer (final jawab), sath mein `memory=True`.

## Step 1 — Environment setup

In [ ]:
!pip install crewai ddgs sentence-transformers pypdf -q

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from google.colab import userdata

llm = LLM(model="groq/openai/gpt-oss-20b", api_key=userdata.get("GROQ_API_KEY"))

## Step 2 — Documents upload aur text extract
`notes.txt` upload karein (ya apni PDF).

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# .txt file ke liye
with open("notes.txt", "r", encoding="utf-8") as f:
    document_text = f.read()

# .pdf ke liye (upar wala hata kar ye use karein):
# from pypdf import PdfReader
# reader = PdfReader("yourfile.pdf")
# document_text = ""
# for page in reader.pages:
#     document_text += page.extract_text()

print(document_text[:300])

## Step 3 — Chunk aur Embed

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(document_text)
print(f"Created {len(chunks)} chunks")

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks)

## Step 4 — Retrieval test (agents ke baghair)

In [ ]:
import numpy as np

def retrieve(query, top_k=3):
    q = embedder.encode([query])[0]
    # cosine similarity
    sims = chunk_embeddings @ q / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q))
    best = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in best]

results = retrieve("What is a merge conflict?")
for r in results:
    print(r, "\n---")

### ✅ COMMIT 1 — `RAG pipeline working — documents chunked, embedded, and retrievable`

## Step 5 — Document Search tool + Researcher

In [ ]:
@tool("Document Search")
def document_search(query: str) -> str:
    """Searches the student's own uploaded documents for relevant passages."""
    results = retrieve(query)
    return str(results)

researcher = Agent(
    role="Personal Notes Researcher",
    goal="Find the passages in the student's own uploaded documents that are most relevant to the question.",
    backstory="You are a careful research assistant who only reads the student's own notes. "
              "You never use the open web and never invent facts that are not in the notes.",
    tools=[document_search],
    llm=llm
)

## Step 6 — Web Search tool + Verifier

In [ ]:
@tool("Web Search")
def web_search(query: str) -> str:
    """Searches the web for current information on a given query."""
    from ddgs import DDGS
    results = DDGS().text(query, max_results=3)
    return str(results)

verifier = Agent(
    role="Fact Verifier",
    goal="Check the claims found in the student's notes against current information on the live web.",
    backstory="You are a skeptical fact-checker. Notes can be outdated or wrong, so you never just agree "
              "with the Researcher. You always search the web and report clearly whether each claim is "
              "confirmed, outdated, or wrong.",
    tools=[web_search],
    llm=llm
)

### ✅ COMMIT 2 — `Researcher and Verifier agents working against your documents and the web`

## Step 7 — Writer (koi tool nahi) aur Crew

In [ ]:
writer = Agent(
    role="Answer Writer",
    goal="Write one clear, accurate final answer using the retrieved notes and the verification result.",
    backstory="You are a clear writer who turns research and fact-check results into a short, "
              "easy-to-understand answer. If the web contradicts the notes, you say so plainly.",
    llm=llm   # NOTE: tools nahi — Writer ka kaam sirf synthesis hai
)

research_task = Task(
    description=(
        "The student asked: '{question}'. Use the Document Search tool to find the passages in the "
        "student's own documents that answer this question. List the relevant passages and the key claims they make."
    ),
    agent=researcher,
    expected_output="A list of relevant passages from the notes and the key factual claims in them."
)

verify_task = Task(
    description=(
        "Take the key claims found by the Researcher for the question '{question}'. "
        "You MUST call the Web Search tool for at least one specific claim and compare the result with the notes. "
        "For each claim say: CONFIRMED, OUTDATED, or WRONG, with a short reason."
    ),
    agent=verifier,
    context=[research_task],
    expected_output="Each claim marked CONFIRMED / OUTDATED / WRONG with a short web-based reason."
)

write_task = Task(
    description=(
        "Write the final answer to the question '{question}' using the Researcher's notes and the "
        "Verifier's result. If the web contradicts the notes, mention the correction clearly."
    ),
    agent=writer,
    context=[verify_task],
    expected_output="One clear final answer in 1-2 short paragraphs."
)

crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential
)

result = crew.kickoff(inputs={"question": "What is a merge conflict and how do I resolve it?"})
print(result.raw)

### ✅ COMMIT 3 — `Full 3-agent crew producing answers end to end`

## Step 8 — Memory on karein

In [ ]:
crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential,
    memory=True,
    embedder={
        "provider": "huggingface",
        "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"}
    }
)

In [ ]:
# Run 1
r1 = crew.kickoff(inputs={"question": "What is RAG and how does it work?"})
print(r1.raw)

In [ ]:
# Run 2 — pehle sawal se related
r2 = crew.kickoff(inputs={"question": "Why do we split documents into chunks in that process?"})
print(r2.raw)

In [ ]:
# Python wali outdated claim ka test (Verifier ko web se correct karna chahiye)
r3 = crew.kickoff(inputs={"question": "What is the latest stable version of Python?"})
print(r3.raw)

### ✅ COMMIT 4 — `Memory enabled — two related runs show the crew recalling context`

Phir README.md likhein aur **COMMIT 5** — `README and sample Q&A added — final submission`